# Agentic equation discovery — Google Colab runner

This notebook runs **`agentic_discovery.py`** on a Colab VM. It does the three Colab-specific things the script can't do on its own:

1. **installs** the LangChain / LangGraph stack,
2. **downloads** `agentic_discovery.py` from the public repo, and
3. **sets your API key** (via Colab *Secrets* or a prompt).

Then it just calls `ad.main()` — the same interactive terminal demo. The dataset prompt and the improve/stop questions appear as **input boxes below the run cell**.

> Run the cells **top to bottom** (nothing to upload). The provider/key cell must run *before* the import cell, because `agentic_discovery.py` reads `LLM_PROVIDER` and the model name at import time.

In [ ]:
# 1) Install the agentic stack. Colab already ships numpy / scipy / matplotlib.
!pip -q install "langgraph>=0.2.55" "langchain-core>=0.3.0" "langchain-google-genai>=2.0.0" "langchain-anthropic>=0.3.0"

In [ ]:
# 2) Get agentic_discovery.py onto the VM (auto-downloaded from the public repo).
import os

REPO_RAW = "https://raw.githubusercontent.com/franciscovillaescusa/AI-Science_Chicago_2026/main"
if not os.path.exists("agentic_discovery.py"):
    !wget -q {REPO_RAW}/agentic_discovery.py

assert os.path.exists("agentic_discovery.py"), "agentic_discovery.py not found on the VM."
print("Found agentic_discovery.py")

# --- Offline / private fork? Upload it manually instead ---
# from google.colab import files
# files.upload()            # choose agentic_discovery.py

In [ ]:
# 3) Choose backend + API key. MUST run BEFORE importing the module.
import os

PROVIDER = "google"   # "google" (Gemini) or "anthropic" (Claude)

KEY_ENV = {"google": "GOOGLE_API_KEY", "anthropic": "ANTHROPIC_API_KEY"}[PROVIDER]
os.environ["LLM_PROVIDER"] = PROVIDER

# Optional: override the model (otherwise the module's per-provider default is used).
# os.environ["LLM_MODEL"] = "gemini-2.5-flash"   # or "claude-haiku-4-5", etc.

# Prefer a Colab Secret (the key icon in the left sidebar); otherwise prompt for it.
if not os.environ.get(KEY_ENV):
    try:
        from google.colab import userdata
        os.environ[KEY_ENV] = userdata.get(KEY_ENV)
    except Exception:
        import getpass
        os.environ[KEY_ENV] = getpass.getpass(f"Enter {KEY_ENV}: ")

assert os.environ.get(KEY_ENV), f"{KEY_ENV} is not set."
print(f"Provider: {PROVIDER}  |  {KEY_ENV} is set")

In [ ]:
# 4) Import (this reads the env vars set above) and run the full demo.
import agentic_discovery as ad
print("Model:", ad.MODEL)

# main() asks you for the hidden formula, then loops Coder -> Executor ->
# Plotter -> Critic and pauses for your improve/stop decision. Type your
# answers in the input boxes that appear just below this cell.
ad.main()

In [ ]:
# 5) main() saved one PNG per iteration under plots/. Show them inline.
import glob
from IPython.display import Image, display

pngs = sorted(glob.glob("plots/fit_iter_*.png"))
for p in pngs:
    print(p)
    display(Image(p))
if not pngs:
    print("No plots yet — run the cell above first.")

### Notes

- **Changing provider or model:** edit `PROVIDER` (or `LLM_MODEL`) in step 3, then **Runtime → Restart session** and re-run from the top — the module captures those values at import time, so a plain re-run won't pick them up.
- **Formulas to try at the prompt:** e.g. `0.6*x + 1.0 + 0.5*sin(2.2*x + 0.5)`, or a Bessel-like `sqrt(2/(pi*sqrt(x**2 + 1)))*cos(x - pi/4)`. Usable names: `exp, log, log10, sqrt, sin, cos, tan, tanh, arcsin, arccos, arctan, abs, sign, pi, e` and `**` for powers.
- **Getting your results:** the generated code lands in `code/` and the plots in `plots/` — open the file browser (folder icon, left sidebar) and right-click to download. Note that each run wipes both folders for a clean start.